In [1]:
# import libraries
import requests
from bs4 import BeautifulSoup
import time
import random
import os
import pandas as pd

In [2]:
def scrape_page(base_url):
    header = { "User-Agent" : "Scrapper misinformation spread for research purpose  pgajbhiye@uchicago.edu" }
    response = requests.get(base_url, headers = header, verify=False)
    soup = BeautifulSoup(response.content, "html.parser")

    # Initialize the list to store the data
    categories = []
    links = []
    base = "https://www.boomlive.in"
    # Find all the relevant divs with class 'col-md-3'
    items = soup.find_all('div', class_='col-md-3')

    # Loop through the divs and extract the data

    for item in items:
        # Extract link
        link_tag = item.find('a', class_='img_link')
        if link_tag:
            link =  base + link_tag['href']
            links.append(link)
        else:
            links.append(None)
        
        # Extract category
        category_tag = item.find('div', class_='category-label')
        if category_tag:
            category = category_tag.text.strip()
            categories.append(category)
        else:
            categories.append(None)
        

    # Create a dictionary to store the data
    data = {
        'Category': categories,
        'Link': links
    }

    return data

In [31]:
# Base URL of the website
base_url = "https://www.boomlive.in/fact-check/"

data = scrape_page(base_url)
df = pd.DataFrame(data)

for i in range(9, 10):
    url = base_url + str(i)
    data = scrape_page(url)
    df = pd.concat([df, pd.DataFrame(data)], ignore_index=True)
    time.sleep(random.randint(1, 3))

C:\Users\prita\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\urllib3\connectionpool.py:1103: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.boomlive.in'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\prita\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\urllib3\connectionpool.py:1103: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.boomlive.in'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [32]:
df.shape

(40, 2)

In [33]:
df.head()

,Category,Link
0,Fact Check,https://www.boomlive.in/fact-check/factcheck-m...
1,Fact Check,https://www.boomlive.in/fact-check/delhi-weddi...
2,Fact Check,https://www.boomlive.in/fact-check/opinion-pol...
3,None,None
4,Fact Check,https://www.boomlive.in/fact-check/false-claim...


In [34]:
# drop rows with None values
df = df.dropna()
df.shape

(30, 2)

In [36]:
df.to_csv('boomlive_links_6.csv', index=False)

In [286]:
# Read the data
df = pd.read_csv('boomlive_links_2.csv')
df.head()

,Category,Link
0,Fact Check,https://www.boomlive.in/fact-check/awami-leagu...
1,Fact Check,https://www.boomlive.in/fact-check/video-of-ma...
2,Fact Check,https://www.boomlive.in/fact-check/lebanon-tv-...
3,Fact Check,https://www.boomlive.in/fact-check/israel-pale...
4,Fact Check,https://www.boomlive.in/fact-check/fact-check-...


In [37]:
df["Category"].value_counts()

Category
Fact Check    29
Fast Check     1
Name: count, dtype: int64

In [38]:
# keep only the rows with category as Fact Check, Politiccs
df = df[df["Category"].isin(["Fact Check", "Politics"])]

In [40]:
df['Category'].value_counts()

Category
Fact Check    29
Name: count, dtype: int64

In [21]:
def scrape_url(url):
    header = { "User-Agent" : "Scrapper misinformation spread for research purpose  pgajbhiye@uchicago.edu" }
    response = requests.get(url, headers = header, verify=False)
    soup = BeautifulSoup(response.content, "html.parser")

    if soup.find('h1', class_='entry-title mb-10 entry-title-main-heading'):
        heading = soup.find('h1', class_='entry-title mb-10 entry-title-main-heading').text.strip()
        sub_heading = soup.find('h2', class_='single-post-summary-heading').text.strip()
        author_date = soup.find('div', class_='meta-details author_dark') #col-md-6
        author = author_date.find('a').text.strip()
        date = soup.find('span', class_='convert-to-localtime').text.strip()
    else:
        heading = soup.find('h1', class_='entry-title mb-10').text.strip()
        sub_heading = soup.find('h2').text.strip()
        author_date = soup.find('div', class_='meta-details author_dark') #col-md-6
        author = author_date.find('a').text.strip()
        date = soup.find('span', class_='convert-to-localtime').text.strip()

    # print(heading)
    # print(sub_heading)
    # print(author)
   #  print(date)

    if soup.find('div', class_='claim-review-block-1'):
        claim_review_block = soup.find('div', class_='claim-review-block-1')
        claim_fact = claim_review_block.find_all('span', class_='value')
        claim = claim_fact[0].text.strip()
        fact_check = claim_fact[1].text.strip()
        # print(claim)
        # print(fact_check)
    elif soup.find('div', class_='col-md-8 article-section'):
        whole_story = soup.find('div', class_='col-md-8 article-section').find_all('p')
        claim = whole_story[0].text.strip()
        # print(claim)
        # the paragraph which starts with "BOOM found that"
        fact_check = [p.text.strip() for p in whole_story if p.text.strip().startswith('BOOM found that')]
        if fact_check:
            fact_check = fact_check[0]
        else:
            fact_check = None
    else:
        whole_story = soup.find('div', class_='details-story-wrapper').find_all('p')
        claim = whole_story[0].text.strip()
        # print(claim)
        # the paragraph which starts with "BOOM found that"
        fact_check = [p.text.strip() for p in whole_story if p.text.strip().startswith('BOOM found that')]
        if fact_check:
            fact_check = fact_check[0]
        else:
            fact_check = None
        # print(fact_check)

    if soup.find('div', class_='claim-review-block mt-40 mb-40'):
        claim_summary = soup.find('div', class_='claim-review-block mt-40 mb-40')
        claimed = claim_summary.find_all('span', class_='value')
        claim_summ = claimed[0].text.strip()
        claimed_by = claimed[1].text.strip()
        fact_check_summ = claimed[2].text.strip()
    else:
        claim_summ = None
        claimed_by = None
        fact_check_summ = None

    # print(claim_summ)
    # print(claimed_by)
    # print(fact_check_summ)

    if soup.find('div', class_='details-story-wrapper'):
        links_post = soup.find('div', class_='details-story-wrapper').find_all('a')
        links = []
        for link in links_post:
            # drop those links which contain 'boomlive' in them
            if 'boomlive' not in link['href']:
                links.append(link['href'])
        # print(links)
    else:
        links = None
    return heading, sub_heading, author, date, claim, fact_check, claim_summ, claimed_by, fact_check_summ, links

In [39]:
# itereate over all the links and scrape the data from df['links]
headings = []
sub_headings = []
authors = []
dates = []
claims = []
fact_checks = []
claim_summs = []
claimed_bys = []
fact_check_summs = []
links = []

for index, l in enumerate(df['Link']):
    print(l)
    heading, sub_heading, author, date, claim, fact_check, claim_summ, claimed_by, fact_check_summ, link = scrape_url(l)
    headings.append(heading)
    sub_headings.append(sub_heading)
    authors.append(author)
    dates.append(date)
    claims.append(claim)
    fact_checks.append(fact_check)
    claim_summs.append(claim_summ)
    claimed_bys.append(claimed_by)
    fact_check_summs.append(fact_check_summ)
    links.append(link)
    time.sleep(random.randint(1, 3))
    # add print statements to check the progress
    print("Completed: ", index)


https://www.boomlive.in/fact-check/factcheck-maha-kumbh-2025-ai-generated-voice-flight-announcement-27712


C:\Users\prita\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\urllib3\connectionpool.py:1103: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.boomlive.in'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Completed:  0
https://www.boomlive.in/fact-check/delhi-wedding-amazon-mx-player-ad-claims-choli-ke-peeche-dance-fact-check-27708


C:\Users\prita\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\urllib3\connectionpool.py:1103: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.boomlive.in'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Completed:  1
https://www.boomlive.in/fact-check/opinion-polls-by-abp-news-aaj-tak-predicting-aap-victory-are-fake-27703


...

Completed:  26
https://www.boomlive.in/fact-check/fact-check-india-business-tycoon-ratan-tata-dies-last-video-fake-news-26703


C:\Users\prita\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\urllib3\connectionpool.py:1103: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.boomlive.in'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Completed:  27
https://www.boomlive.in/fact-check/factcheck-india-united-nations-security-council-veto-power-permanent-member-26694


C:\Users\prita\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\urllib3\connectionpool.py:1103: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.boomlive.in'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Completed:  28


In [23]:
len(headings)

95

In [43]:
data = df.copy()

In [44]:
# add the data to the dataframe
data['Heading'] = headings
data['Sub_heading'] = sub_headings
data['Author'] = authors
data['Date'] = dates
data['Claim'] = claims
data['Fact_check'] = fact_checks
data['Claim_summary'] = claim_summs
data['Claimed_by'] = claimed_bys
data['Fact_check_summary'] = fact_check_summs
data['Links'] = links

In [45]:
data.shape

(29, 12)

In [46]:
data.tail()

,Category,Link,Heading,Sub_heading,Author,Date,Claim,Fact_check,Claim_summary,Claimed_by,Fact_check_summary,Links
33,Fact Check,https://www.boomlive.in/fact-check/sunita-will...,Sunita Williams Returning From Space? Old Vide...,"Since June 2024, Williams has been aboard the ...",Archis Chowdhury,14 Oct 2024 2:20 PM IST,A viral video claims that astronaut Sunita Wil...,The video going viral is from November 2012. S...,Video shows NASA astronaut Sunita Williams ret...,Social media users,False,"[https://archive.is/77tsX, https://www.faceboo..."
34,Fact Check,https://www.boomlive.in/fact-check/fact-check-...,Video of Bangladeshi TikToker Attacked For All...,The incident is from Chittagong's Cox's Bazar ...,Tausif Akbar,14 Oct 2024 11:41 AM IST,Video shows a Hindu woman being beaten in Bang...,The woman in the viral video is a Bangladeshi ...,Video shows a Hindu woman being beaten in Bang...,X and Facebook Users,False,[https://x.com/Warlock_Shabby/status/184298982...
36,Fact Check,https://www.boomlive.in/fact-check/fake-news-a...,News Outlets Run AI-Generated Photo Showing Ra...,BOOM found that the viral black-and-white phot...,Anmol Alphonso,11 Oct 2024 6:43 PM IST,A black-and-white photo circulating online dep...,BOOM found that the viral black-and-white phot...,Old black-and-white photo shows a young Ratan ...,"News 18 Hindi, Indiatimes",False,"[https://perma.cc/9PFN-QQB7?type=standard, htt..."
37,Fact Check,https://www.boomlive.in/fact-check/fact-check-...,Media Outlets Misreport Unrelated Videos As Ra...,BOOM found that Ratan Tata attended several ev...,Srijanee Chakraborty,11 Oct 2024 5:53 PM IST,Two viral videos circulating online show the l...,The viral videos are old and does not show Rat...,Two viral videos circulating online show the l...,News Outlets and Social Media Users,False,[https://www.facebook.com/reel/559799483382090...
38,Fact Check,https://www.boomlive.in/fact-check/factcheck-i...,Claims Of India Becoming A Permanent Member Of...,"As per UN's official website, India has not ye...",Nidhi Jacob,10 Oct 2024 2:22 PM IST,India is granted with a permanent membership i...,This claim is false. BOOM checked the official...,India is granted with a permanent membership i...,"X users, Facebook users",False,[https://www.mea.gov.in/Speeches-Statements.ht...


In [296]:
data.shape

(1646, 12)

In [47]:
data.to_csv('boomlive_data_7.csv', index=False)

In [48]:
#read the data
data1 = pd.read_csv('boomlive_data_6.csv')

In [49]:
data1.head()

,Category,Link,Heading,Sub_heading,Author,Date,Claim,Fact_check,Claim_summary,Claimed_by,Fact_check_summary,Links
0,Fact Check,https://www.boomlive.in/fact-check/opinion-pol...,"Opinion Polls By ABP News, Aaj Tak Predicting ...",BOOM found that news bulletins of Aaj Tak and ...,Swasti Chatterjee,5 Feb 2025 1:32 PM IST,Opinion polls by ABP News and Aaj Tak show a l...,ABP News and India Today Group have denied con...,Opinion polls by ABP News and Aaj Tak show a l...,Unknown,False,['https://archive.is/https://x.com/SakshiGupta...
1,Fact Check,https://www.boomlive.in/fact-check/false-claim...,India Today Graphic Predicting A BJP Win in De...,India Today issued a statement clarifying that...,Srijanee Chakraborty,4 Feb 2025 6:39 PM IST,Pre-poll survey conducted by India Today predi...,The viral graphics are fake. India Today group...,Graphics show pre-poll survey conducted by Ind...,Facebook and X users,False,['https://www.facebook.com/permalink.php?story...
2,Fact Check,https://www.boomlive.in/fact-check/shah-rukh-k...,AI Photos Viral As Shah Rukh Khan With WWE Wre...,BOOM tested the images using an AI-image dete...,Srijit Das,3 Feb 2025 3:20 PM IST,Photos show Shah Rukh Khan with American wrest...,The photographs do not show any real visual; t...,Photos show Shah Rukh Khan with American wrest...,Social Media Users,False,['https://www.facebook.com/permalink.php?story...
3,Fact Check,https://www.boomlive.in/fact-check/strawberry-...,Return Of The Viral 'Strawberry Quick Meth' Hoax,BOOM found that the Strawberry Quick claim is ...,Archis Chowdhury,1 Feb 2025 5:31 PM IST,The claim suggests 'Strawberry Quick' is a for...,The claim is false; it is a debunked hoax from...,Image shows a dangerous flavoured methamphetam...,"X, Facebook and Instagram users",False,"['https://perma.cc/7229-7EJY', 'https://www.fa..."
4,Fact Check,https://www.boomlive.in/fact-check/factcheck-e...,Viral Post Claiming Egypt Preparing To Introdu...,BOOM found that the hoax is old and there has ...,Nidhi Jacob,31 Jan 2025 1:42 PM IST,Egypt’s new Islamist dominated parliament is p...,BOOM reached out to fact-checking organisation...,Egypt’s parliament is preparing to introduce a...,https://x.com/JaipurDialogues/status/188312704...,False,"['https://t.co/1LUKj1bhex', 'https://twitter.c..."


In [ ]:
# merge both the data and drop duplicates
data = pd.concat([data, data1], ignore_index=True)
data.shape



(124, 12)

In [53]:
data.head()

,Category,Link,Heading,Sub_heading,Author,Date,Claim,Fact_check,Claim_summary,Claimed_by,Fact_check_summary,Links
0,Fact Check,https://www.boomlive.in/fact-check/factcheck-m...,AI Generated Audio Viral Claiming An Airline P...,"The video was originally taken with a drone, a...",Jagriti Trisha,6 Feb 2025 4:33 PM IST,"While landing in Prayagraj, the pilot of an in...","The video was originally taken with a drone, a...",None,None,None,"[https://ghostarchive.org/archive/dHnq4, https..."
1,Fact Check,https://www.boomlive.in/fact-check/delhi-weddi...,Delhi Wedding Cancelled Over 'Choli Ke Peeche'...,BOOM found that news is not real and appeared ...,Srijit Das,5 Feb 2025 7:34 PM IST,Delhi wedding called off after groom's dance t...,The reports are fake and do not describe any r...,Delhi wedding called off after groom's dance t...,"The Times of India, The Indian Express, The Ec...",False,"[https://ghostarchive.org/archive/6IWD1, https..."
2,Fact Check,https://www.boomlive.in/fact-check/opinion-pol...,"Opinion Polls By ABP News, Aaj Tak Predicting ...",BOOM found that news bulletins of Aaj Tak and ...,Swasti Chatterjee,5 Feb 2025 1:32 PM IST,Opinion polls by ABP News and Aaj Tak show a l...,ABP News and India Today Group have denied con...,Opinion polls by ABP News and Aaj Tak show a l...,Unknown,False,[https://archive.is/https://x.com/SakshiGupta_...
3,Fact Check,https://www.boomlive.in/fact-check/false-claim...,India Today Graphic Predicting A BJP Win in De...,India Today issued a statement clarifying that...,Srijanee Chakraborty,4 Feb 2025 6:39 PM IST,Pre-poll survey conducted by India Today predi...,The viral graphics are fake. India Today group...,Graphics show pre-poll survey conducted by Ind...,Facebook and X users,False,[https://www.facebook.com/permalink.php?story_...
4,Fact Check,https://www.boomlive.in/fact-check/shah-rukh-k...,AI Photos Viral As Shah Rukh Khan With WWE Wre...,BOOM tested the images using an AI-image dete...,Srijit Das,3 Feb 2025 3:20 PM IST,Photos show Shah Rukh Khan with American wrest...,The photographs do not show any real visual; t...,Photos show Shah Rukh Khan with American wrest...,Social Media Users,False,[https://www.facebook.com/permalink.php?story_...


In [54]:
#drop duplicates
data = data.drop_duplicates(subset=['Heading'], keep='first')
data.shape

(111, 12)

In [55]:
# convert the date column to datetime
data['Date'] = pd.to_datetime(data['Date'])

C:\Users\prita\AppData\Local\Temp\ipykernel_8228\1814344273.py:2: FutureWarning: Parsed string "6 Feb 2025 4:33 PM IST" included an un-recognized timezone "IST". Dropping unrecognized timezones is deprecated; in a future version this will raise. Instead pass the string without the timezone, then use .tz_localize to convert to a recognized timezone.
  data['Date'] = pd.to_datetime(data['Date'])
C:\Users\prita\AppData\Local\Temp\ipykernel_8228\1814344273.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data['Date'] = pd.to_datetime(data['Date'])
C:\Users\prita\AppData\Local\Temp\ipykernel_8228\1814344273.py:2: FutureWarning: Parsed string "5 Feb 2025 7:34 PM IST" included an un-recognized timezone "IST". Dropping unrecognized timezones is deprecated; in a future version this will raise. Instead pass the string without the timezone, then use .tz_lo

In [56]:
data.head()

,Category,Link,Heading,Sub_heading,Author,Date,Claim,Fact_check,Claim_summary,Claimed_by,Fact_check_summary,Links
0,Fact Check,https://www.boomlive.in/fact-check/factcheck-m...,AI Generated Audio Viral Claiming An Airline P...,"The video was originally taken with a drone, a...",Jagriti Trisha,2025-02-06 16:33:00,"While landing in Prayagraj, the pilot of an in...","The video was originally taken with a drone, a...",None,None,None,"[https://ghostarchive.org/archive/dHnq4, https..."
1,Fact Check,https://www.boomlive.in/fact-check/delhi-weddi...,Delhi Wedding Cancelled Over 'Choli Ke Peeche'...,BOOM found that news is not real and appeared ...,Srijit Das,2025-02-05 19:34:00,Delhi wedding called off after groom's dance t...,The reports are fake and do not describe any r...,Delhi wedding called off after groom's dance t...,"The Times of India, The Indian Express, The Ec...",False,"[https://ghostarchive.org/archive/6IWD1, https..."
2,Fact Check,https://www.boomlive.in/fact-check/opinion-pol...,"Opinion Polls By ABP News, Aaj Tak Predicting ...",BOOM found that news bulletins of Aaj Tak and ...,Swasti Chatterjee,2025-02-05 13:32:00,Opinion polls by ABP News and Aaj Tak show a l...,ABP News and India Today Group have denied con...,Opinion polls by ABP News and Aaj Tak show a l...,Unknown,False,[https://archive.is/https://x.com/SakshiGupta_...
3,Fact Check,https://www.boomlive.in/fact-check/false-claim...,India Today Graphic Predicting A BJP Win in De...,India Today issued a statement clarifying that...,Srijanee Chakraborty,2025-02-04 18:39:00,Pre-poll survey conducted by India Today predi...,The viral graphics are fake. India Today group...,Graphics show pre-poll survey conducted by Ind...,Facebook and X users,False,[https://www.facebook.com/permalink.php?story_...
4,Fact Check,https://www.boomlive.in/fact-check/shah-rukh-k...,AI Photos Viral As Shah Rukh Khan With WWE Wre...,BOOM tested the images using an AI-image dete...,Srijit Das,2025-02-03 15:20:00,Photos show Shah Rukh Khan with American wrest...,The photographs do not show any real visual; t...,Photos show Shah Rukh Khan with American wrest...,Social Media Users,False,[https://www.facebook.com/permalink.php?story_...


In [57]:
# extract year and month from the date column
data['Year'] = data['Date'].dt.year
data['Month'] = data['Date'].dt.month

C:\Users\prita\AppData\Local\Temp\ipykernel_8228\221706629.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Year'] = data['Date'].dt.year
C:\Users\prita\AppData\Local\Temp\ipykernel_8228\221706629.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Month'] = data['Date'].dt.month


In [58]:
data.head()

,Category,Link,Heading,Sub_heading,Author,Date,Claim,Fact_check,Claim_summary,Claimed_by,Fact_check_summary,Links,Year,Month
0,Fact Check,https://www.boomlive.in/fact-check/factcheck-m...,AI Generated Audio Viral Claiming An Airline P...,"The video was originally taken with a drone, a...",Jagriti Trisha,2025-02-06 16:33:00,"While landing in Prayagraj, the pilot of an in...","The video was originally taken with a drone, a...",None,None,None,"[https://ghostarchive.org/archive/dHnq4, https...",2025,2
1,Fact Check,https://www.boomlive.in/fact-check/delhi-weddi...,Delhi Wedding Cancelled Over 'Choli Ke Peeche'...,BOOM found that news is not real and appeared ...,Srijit Das,2025-02-05 19:34:00,Delhi wedding called off after groom's dance t...,The reports are fake and do not describe any r...,Delhi wedding called off after groom's dance t...,"The Times of India, The Indian Express, The Ec...",False,"[https://ghostarchive.org/archive/6IWD1, https...",2025,2
2,Fact Check,https://www.boomlive.in/fact-check/opinion-pol...,"Opinion Polls By ABP News, Aaj Tak Predicting ...",BOOM found that news bulletins of Aaj Tak and ...,Swasti Chatterjee,2025-02-05 13:32:00,Opinion polls by ABP News and Aaj Tak show a l...,ABP News and India Today Group have denied con...,Opinion polls by ABP News and Aaj Tak show a l...,Unknown,False,[https://archive.is/https://x.com/SakshiGupta_...,2025,2
3,Fact Check,https://www.boomlive.in/fact-check/false-claim...,India Today Graphic Predicting A BJP Win in De...,India Today issued a statement clarifying that...,Srijanee Chakraborty,2025-02-04 18:39:00,Pre-poll survey conducted by India Today predi...,The viral graphics are fake. India Today group...,Graphics show pre-poll survey conducted by Ind...,Facebook and X users,False,[https://www.facebook.com/permalink.php?story_...,2025,2
4,Fact Check,https://www.boomlive.in/fact-check/shah-rukh-k...,AI Photos Viral As Shah Rukh Khan With WWE Wre...,BOOM tested the images using an AI-image dete...,Srijit Das,2025-02-03 15:20:00,Photos show Shah Rukh Khan with American wrest...,The photographs do not show any real visual; t...,Photos show Shah Rukh Khan with American wrest...,Social Media Users,False,[https://www.facebook.com/permalink.php?story_...,2025,2


In [59]:
# drop rows with year == 2025
data = data[data['Year'] != 2025]
data.shape


(93, 14)

In [61]:
#soert the data by date
data = data.sort_values(by='Date', ascending=False)

In [62]:
data.head()

,Category,Link,Heading,Sub_heading,Author,Date,Claim,Fact_check,Claim_summary,Claimed_by,Fact_check_summary,Links,Year,Month
45,Fact Check,https://www.boomlive.in/fact-check/dekh-rahe-h...,"""Dekh Rahe Ho Binod?"" AAP And BJP Share Deepfa...",The videos have been manipulated using AI voic...,Swasti Chatterjee,2024-12-31 17:35:00,The Bharatiya Janata Party (BJP) and Aam Aadmi...,NaN,NaN,NaN,NaN,"['https://perma.cc/JHF8-FY26', 'https://t.co/E...",2024,12
46,Fact Check,https://www.boomlive.in/fact-check/video-son-m...,Viral Posts Falsely Claim Son Married His Moth...,BOOM found that the visuals in the posts are f...,Anmol Alphonso,2024-12-31 15:49:00,Video shows Pakistani boy marrying his mother,A Pakistani youngster Abdul Ahad posted a hear...,Video shows Pakistani boy marrying his mother,Social media posts,False,['https://x.com/Knight491656903/status/1873663...,2024,12
47,Fact Check,https://www.boomlive.in/fact-check/couple-publ...,Video Of Couple Beaten By Henchman In WB Peddl...,"The incident happened in Chopra, West Bengal i...",Anmol Alphonso,2024-12-31 11:50:00,Video shows a couple being publicly flogged in...,"The video is from Chopra, West Bengal, and not...",Video shows a couple being publicly flogged in...,Social media posts,False,['https://indianexpress.com/article/world/bang...,2024,12
48,Fact Check,https://www.boomlive.in/fact-check/viral-image...,Viral Photo Does Not Show Manmohan Singh Touch...,The image is from 2011 and shows a representat...,Srijanee Chakraborty,2024-12-30 15:43:00,Viral image shows former Indian Prime Minister...,The man touching Sonia Gandhi's feet is a repr...,Viral image shows former Indian Prime Minister...,Social Media Users,False,"['https://t.co/43ZyfojnWt', 'https://twitter.c...",2024,12
49,Fact Check,https://www.boomlive.in/fact-check/viral-video...,Video Does Not Show Hindu Woman Raped And Tort...,BOOM found that the woman in the video is a Mu...,Tausif Akbar,2024-12-29 14:04:00,Video shows a Hindu woman who was raped and to...,A local journalist confirmed to BOOM Banglades...,Video shows a Hindu woman who was raped and to...,Social Media Users,False,['https://x.com/Asifurrahman71/status/18725269...,2024,12


In [64]:
data.to_csv('boomlive_data_8.csv', index=False)